<center><h1> FIL2000 - Ética Aplicada a Inteligencia Artificial </h1><center>
<center><h2> Taller Sumativo 1: Modelos interpretables y xAI <h2><center>  




# `Instrucciones generales`
Completa las partes de este notebook y escribe un informe que contenga tus resultados.


## Estructura del informe

| Sección | Contenido esperado | Puntaje |
|---|---|---|
| **Introducción** | Contextualiza brevemente el problema explicando los conceptos clave: XAI, LIME y SHAP. | 10% |
| **Parte I — LIME** | Carga el modelo BETO, define las frases de interés y aplica LIME. Visualiza e interpreta las contribuciones de cada token y responde las preguntas guiadas. | 25% |
| **Parte II — SHAP** | Entrena el Random Forest, calcula los SHAP values y visualiza las variables más influyentes. Interpreta qué features determinan las predicciones y por qué eso es relevante en el contexto del problema. | 25% |
| **Análisis comparativo** | Compara LIME y SHAP técnicamente considerando: tipo de interpretabilidad (local o global), supuestos y limitaciones. Apoya la comparación con ejemplos concretos de tus propios resultados. finalmente responde ¿Qué tipo de explicaciones podríamos generar con esta información? | 25% |
| **Código** | El código es legible, está bien implementado y cuenta con documentación mínima. | 15% |

## `Temas formales`

**Modalidad:** Debe realizarse de manera individual

**Formato de entrega:** El trabajo debe elaborarse y entregarse a través de Canvas. Tanto el Jupyter como el informe deben entregarse en formato PDF.

**Extensión:** Máximo 4 planas.

**Formato del documento:**
- Citas: APA 7
- Formato: dos columnas
- Fuente: Times New Roman, tamaño 12
- Interlineado: 1,5

**No es necesario justificar resultados o métodos o responder preguntas directamente en el Jupyter**, pero el informe sí debe incluir los resultados y las reflexiones planteadas en él. Toda métrica o visualización que no esté acompañada de una explicación no será considerada en la evaluación.

## `Integridad académica y uso de IA`

El uso de asistentes de IA para apoyarse con código no está prohibido, pero debe ser documentado. Pueden usar IA para entender mejor un código o concepto, resolver dudas de sintaxis, identificar y entender bugs, y buscar información o referencias. No pueden usar IA para generar el código completo del taller, reemplazar su razonamiento en decisiones importantes de la tarea, definir automáticamente qué limpiar, normalizar o estandarizar en los datos, determinar la secuencia de pasos de su solución, ni decidir la estrategia de optimización del código. La redacción del informe sigue los lineamientos de integridad académica del curso.

No está permitida la copia completa o parcial entre distintos estudiantes. Cualquier uso no documentado o irresponsable de IA, o plagio, será considerado una falta a la integridad académica.

Si tienen dudas sobre el uso de IA o la citación de código externo, no duden en preguntar. Pueden usar libremente los códigos de los talleres formativos.


## `Importante`

**Lean la rúbrica antes de escribir su informe.**


In [ ]:
# Coloque aquí todas las importaciones que considere necesarias.
from transformers import pipeline
from lime.lime_text import LimeTextExplainer
import numpy as np
import matplotlib.pyplot as plt


# Parte 1: Interpretabilidad local con LIME

En esta parte aplicarás la técnica LIME para obtener explicaciones locales de un modelo de análisis de sentimiento en español.

## Parte 1.1 :

Contexto y motivación (definir antes de escribir código)
Decide brevemente el contexto de tu análisis:

1. ¿Qué tipo de frases vas a analizar? (comentarios de redes sociales, titulares de noticias, reseñas, frases con estereotipos, etc.). Crea o recopila al menos 10 frases.

2. ¿Qué quieres descubrir o investigar con LIME? Por ejemplo: ¿el modelo clasifica correctamente el sarcasmo?, ¿existen sesgos de género en las predicciones?, ¿cómo se comporta ante lenguaje coloquial chileno?


## Parte 1.2:

Carga el modelo BETO de análisis de sentimientos.

Obtén la predicción del modelo para cada frase e identifica si los resultados son consistentes con lo que esperabas.
Aplica LIME para obtener la explicación local de al menos 3 frases. Para cada una: visualiza las contribuciones de cada palabra con un gráfico de barras.

Interpreta los resultados: ¿qué palabras están dominando cada predicción? ¿Hay palabras que no esperabas que tuvieran ese peso? ¿Las explicaciones son coherentes con el significado de la frase?

# Parte 2 : Explicabilidad de Random Forest con SHAP

En esta segunda parte del taller se trabajará con datos del área médica, en específico con lo que respecta a la enfermedad real crónica (CDK). Para entrenar un modelo de Random Forest, aplicando SHAP para conocer los factores/valores que explican los resultados.

## Datos

El dataset a utilizar proviene desde el repositorio de UCI Machine Learning Repository. Se dejan a continuación algunos metadatos:

- Entradas: 400
- Variables :
- Faltan valores : Sí

Las variables que se encuentran especificadas son :

| variable | definición | unidad |
|----------|----------|----------|
|age| Edad| año|
|bp| Presión Sanguínea| mm/Hg
|sg | Densidad relativa|
|al| Albúmina|
|su| Azúcar|
|rbc| Glóbulos rojos|
|pc| Células de pus|
|pcc| Cúmulo de células de pus|
|ba| Bacterias|
|bgr |Glucosa aleatoria| mgs/dl
|bu| Urea Sanguínea| mgs/dl
|sc| Creatinina sérica| mgs/dl
|sod| Sodio| mEq/L|
|pot| Potasio| mEq/L|
|hemo| Hemoglobina| mEq/L|
|pcv| Volumen de células empaquetadas|
|wbcc| Cantidad de Glóbulos Blancos| cells/cmm.|
|rbcc| Cantidad de Glóbulos rojos| millions/cmm|  
|htn| Hipertensión |  
|dm| Diabetes mellitus|
|cad| Enfermedades Arterias Coronarias|
|appet| Apetito|
|pe| Edema Podal|
|ane| Anemia|
|class (TARGET)| ckd / no ckd|



## `Importante` : Acceso a los datos

Puedes revisar el repositorio fuente aquí :  https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease

- Por comodidad se recomienda importar el dataset, ya que los archivos originales están en tipo ".arff"
- Puedes trabajar cargando los archivos manualmente o importanto los datos. A continuación te dejamos la importación en python entregada por el repositorio oficial.

## `Librerias`

In [ ]:
pip install ucimlrepo --quiet

In [ ]:
import pandas as pd
import numpy as np
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMa
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler,MinMaxScaler


In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
chronic_kidney_disease = fetch_ucirepo(id=336)

# data (as pandas dataframes)
X = chronic_kidney_disease.data.features
y = chronic_kidney_disease.data.targets

# metadata
print(chronic_kidney_disease.metadata)

# variable information
print(chronic_kidney_disease.variables)

### Fuente :
# Rubini, L., Soundarapandian, P., & Eswaran, P. (2015). Chronic Kidney Disease [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5G020.

2.1 Carga y limpieza de datos
- Cargue la base de datos y cree el dataframe (En caso de trabajar con el archivo descargado)
- Revise que columnas tienen nulos/datos faltantes. Comente su elección al respecto considerando el contexto de los datos. Manéjelos con : 0, botar fila/columna o imputación de media. Si tiene otro método, indíquelo y justifique por qué lo cree apropiado
- Revise el tipo de dato del dataset. Comente si realiza cambios.
- Codifique las variables categóricas. Puede usar método label encoder, onehotencoder, dummies. Puedes mappear manualmente si es binaria la columna.


2.2 División del conjunto de datos
- En caso de haber cargado el dataset desde un archivo, separe del conjunto total los predictores y la variable objetivo `class`.
- Utilice train_test_split para obtener el conjunto de prueba y entrenamiento.



2.3 Estandarización
- Estandarice los datos numéricos con StandarScaler o Minmax
- Justifique brevemente su elección






2.4 Clasificador Random Forest y métrica de evaluación
- Entrena un Random Forest, selecciona un número de estimadores y entrega las predicciones del modelo.
- Comenta los resultados de la matriz de confusión y el reporte de clasificación. Explica las consecuencias de aplicar ese modelo en área de la salud y en prevención de CDK.

2.5 Resultado y explicaciones con SHAP
- Comenta los beneficios y perjuicios que puede tener una herramienta como SHAP, en general y aterriza al caso de los datos trabajados
- Obten los valores shap.Visualiza el summary plot. Comenta los resultados.
- Analiza que features son importantes para la predicción y que puede influir en que estas tengan mayor influencia. Considera en tu explicación la relevancia de los resultados en tanto el contexto de diagnóstico médico y prevención de problemas de salud.
- Preguntas guía para tu respuesta : ¿Qué aporta conocer los valores SHAP? ¿Es una herramienta suficiente en sí misma o puede apoyarse con otras? ¿Cómo afecta la preparación de los datos a las predicciones y su posterior explicación? ¿Es importante tener acceso a ambas interpretaciónes (global y local) en este caso?